# Zemli: train the crop-check model (tomato / cucumber / coriander / other)

This model runs **before** the disease model. If someone photographs the wrong plant, a hand, soil or anything else, the app says so instead of guessing a disease.

**Runtime → Change runtime type → T4 GPU**, then run the cells **one at a time** (not "Run all"), waiting for the green tick each time. Every cell is safe to run again if something goes wrong.

Colab deletes all files when the runtime disconnects (idle for a long time, tab closed, or overnight). If you come back later and see "No such file or directory", start again from the top.

Where the four classes come from:
- **tomato**: PlantVillage tomato + Bangladesh field-dataset tomato
- **cucumber**: Bangladesh field-dataset cucumber
- **other**: other crops' leaves plus everyday non-leaf photos (Imagenette)
- **coriander**: only if you upload photos in step 5

In [ ]:
!nvidia-smi -L
!pip -q install onnx datasets
import re, io, random, shutil, subprocess
from pathlib import Path
from collections import defaultdict
from PIL import Image
sources = {}   # step name -> {class: [images]}; re-running a step replaces its own entry

# Look-alike crops (squash, bitter gourd, bottle gourd) teach the model to reject real cucumber leaves
# as long as all our cucumber photos are on black backgrounds. Keep this False until you have
# real field photos of cucumber (step 5), then try True.
INCLUDE_CUCURBIT_LOOKALIKES = False
LOOKALIKE = re.compile(r"squash|gourd|pumpkin|melon", re.I)

## 1. Upload the Zemli zip
Any name is fine (for example `zemli-prototype (1).zip`). The last line must say `True`.

In [ ]:
from google.colab import files
import glob, os, subprocess
up = files.upload()
# works whatever the file is called, e.g. "zemli-prototype (1).zip"
name = list(up)[0] if up else sorted(glob.glob("zemli-prototype*.zip"), key=os.path.getmtime)[-1]
subprocess.run(["unzip", "-q", "-o", name], check=True)
print("unzipped:", name)
print("train.py is the latest version:", '"gate"' in open("zemli/training/train.py").read())

## 2. PlantVillage: tomato leaves + many other crops' leaves (a few minutes)

In [ ]:
if not Path("pv/.git").exists():   # skip the download if it's already here
    shutil.rmtree("pv", ignore_errors=True)
    subprocess.run("git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git pv", shell=True, check=True)
folders = subprocess.run(["git", "-C", "pv", "ls-tree", "--name-only", "HEAD", "raw/color/"], capture_output=True, text=True).stdout.split("\n")
folders = [f for f in folders if f.strip()]
subprocess.run(["git", "-C", "pv", "sparse-checkout", "set", "--no-cone"] + [f + "/" for f in folders], check=True)

random.seed(42)
TOMATO_PV, OTHER_PER_FOLDER = 2000, 120
part = defaultdict(list)
tomato_files = [p for f in folders if "Tomato" in f for p in Path("pv", f).glob("*.*")]
random.shuffle(tomato_files)
part["tomato"] = tomato_files[:TOMATO_PV]
for f in folders:
    if "Tomato" in f: continue
    imgs = sorted(Path("pv", f).glob("*.*")); random.shuffle(imgs)
    part["other"] += imgs[:OTHER_PER_FOLDER]
sources["plantvillage"] = part
print({k: len(v) for k, v in part.items()})

## 2b. PlantDoc: real field photos (very important)

PlantVillage and the Bangladesh photos are leaves on plain backgrounds. Without real field photos, the model learns "plain background = tomato, busy background = other" and rejects real tomato leaves on the plant. PlantDoc has leaves photographed in real fields and gardens, for tomato and many other crops.

In [ ]:
if not Path("plantdoc/.git").exists():
    shutil.rmtree("plantdoc", ignore_errors=True)
    subprocess.run("git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git plantdoc", shell=True, check=True)
part = defaultdict(list)
for d in sorted(Path("plantdoc").glob("t*/*")):          # train/ and test/ folders
    if not d.is_dir(): continue
    imgs = [p for p in d.glob("*.*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    if LOOKALIKE.search(d.name) and not INCLUDE_CUCURBIT_LOOKALIKES:
        continue
    if d.name.lower().startswith("tomato"):
        part["tomato"] += imgs * 3       # repeated: field photos of tomato are the most important examples
    else:
        part["other"] += imgs            # potato, pepper, squash, grape... photographed in the field
sources["plantdoc"] = part
print({k: len(v) for k, v in part.items()})

## 3. Bangladesh field dataset: cucumber, tomato, and look-alike crops (about 1 GB)

In [ ]:
from datasets import load_dataset
bd = load_dataset("Project-AgML/plant_leaf_disease_classification", split="train")
plants = bd.features["plant_type"].names
print("plant types:", plants)
by_plant = defaultdict(list)
for i, p in enumerate(bd["plant_type"]):
    by_plant[plants[p]].append(i)
random.seed(43)
OTHER_PER_PLANT = 500
part = defaultdict(list)
for name, idxs in by_plant.items():
    random.shuffle(idxs)
    n = name.lower()
    if "cucumber" in n:   part["cucumber"] += [("bd", i) for i in idxs]
    elif "tomato" in n:   part["tomato"] += [("bd", i) for i in idxs]
    elif LOOKALIKE.search(name) and not INCLUDE_CUCURBIT_LOOKALIKES:
        print("leaving out look-alike:", name)
    else:                 part["other"] += [("bd", i) for i in idxs[:OTHER_PER_PLANT]]
sources["bangladesh"] = part
print({k: len(v) for k, v in part.items()})

## 4. Everyday non-leaf photos (so a hand, a bag or a wall is rejected too)

In [ ]:
NON_LEAF = 1500
inet = None
try:
    url = "https://huggingface.co/datasets/frgfm/imagenette/resolve/refs%2Fconvert%2Fparquet/160px/train/0000.parquet"
    inet = load_dataset("parquet", data_files=url, split="train")
    random.seed(44)
    idxs = list(range(len(inet))); random.shuffle(idxs)
    sources["imagenette"] = {"other": [("inet", i) for i in idxs[:NON_LEAF]]}
    print("added", min(NON_LEAF, len(idxs)), "non-leaf photos")
except Exception as e:
    print("Could not load Imagenette, continuing without it:", e)

## 5. (Optional but best) Upload your own photos

Make a zip with folders named exactly `tomato`, `cucumber`, `coriander` and/or `other`, and put photos inside them. Take the photos the way field staff will: leaf on the plant, phone camera, real light.

- `tomato` / `cucumber` / `coriander`: real leaves of that crop, on the plant
- For cucumber, you can also add the Cucumber Disease Recognition Dataset from Mendeley (DOI 10.17632/y6d3z6f8z9.1): put its **leaf** photos (not fruit, not the augmented folder) in the `cucumber` folder. These are real field photos, which this model needs most.
- `other`: things staff might photograph by mistake: weeds, soil, drip pipes, net, hands, nearby plants

Even 50 photos per folder helps a lot. Folders can be inside other folders; only the folder name matters. **If you have none, don't run this cell.**

In [ ]:
up = files.upload()
shutil.rmtree("own_upload", ignore_errors=True)
for name in up:
    shutil.unpack_archive(name, "own_upload")
part = defaultdict(list)
for p in Path("own_upload").rglob("*"):
    if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}: continue
    cls = next((c for c in ["tomato", "cucumber", "coriander", "other"] if c in [x.lower() for x in p.parts]), None)
    if cls: part[cls].append(p)
part = {k: v * 3 for k, v in part.items()}   # repeated so your real farm photos carry more weight
sources["own_upload"] = part
print({k: len(v) // 3 for k, v in part.items()}, "photos found")

## 6. Combine, split 80 / 10 / 10 and write image folders (5-10 minutes)

In [ ]:
from collections import Counter
def load(item):
    if isinstance(item, Path): return Image.open(item)
    src, i = item
    if src == "bd": return bd[i]["image"]
    img = inet[i]["image"]
    return Image.open(io.BytesIO(img["bytes"])) if isinstance(img, dict) else img

pool = defaultdict(list)
for step, part in sources.items():
    for cls, items in part.items():
        pool[cls] += items
print("using:", list(sources))

random.seed(42)
dst = Path("data/gate"); shutil.rmtree(dst, ignore_errors=True)
for cls, items in sorted(pool.items()):
    weight = Counter(items)                 # repeated items = extra weight, but only in training
    unique = sorted(weight, key=str); random.shuffle(unique)
    if len(unique) < 50:
        print(f"skipping {cls}: only {len(unique)} images"); continue
    n = len(unique); a, b = int(n * 0.8), int(n * 0.9)
    for split, chunk in [("train", unique[:a]), ("val", unique[a:b]), ("test_public", unique[b:])]:
        d = dst / split / cls; d.mkdir(parents=True, exist_ok=True)
        for j, it in enumerate(chunk):
            try:
                img = load(it).convert("RGB")
                copies = weight[it] if split == "train" else 1   # val/test never contain duplicates
                for r in range(copies): img.save(d / f"{j}_{r}.jpg", quality=90)
            except Exception as e: print("skip one image:", e)
    print(f"{cls:10s} {n:6d} unique -> train {a}, val {b-a}, test {n-b}")

## 7. Train (about 20-30 minutes)
Wait until it prints `Saved out/gate/model.onnx` before going on.

In [ ]:
import os
print("1. train.py is the latest version:", '"gate"' in open("zemli/training/train.py").read())
print("2. data folders:", os.listdir("data/gate/train") if os.path.exists("data/gate/train") else "MISSING - run steps 1 to 6 first")
!python zemli/training/train.py --crop gate --data data/gate --epochs 6 --batch 64 --workers 2 --out out

## 8. Results
Look at the confusion matrix too: mistakes between `tomato` and `other` usually mean potato or pepper leaves.

In [ ]:
import json
m = json.load(open("out/gate/metrics.json"))
print("classes:", m["classes"])
for split in ["val", "test_public"]:
    print(f"{split:12s} accuracy {m[split]['accuracy']:.4f}  macro-F1 {m[split]['macro_f1']:.4f}")
for c, v in m["test_public"]["per_class"].items(): print(f"  {c:10s} F1 {v['f1']:.3f}  ({v['support']} images)")
print("\nconfusion matrix (rows = true, columns = predicted):", m["classes"])
for c, row in zip(m["classes"], m["test_public"]["confusion_matrix"]): print(f"  {c:10s} {row}")

## 9. Download
Unzip into `backend\models\` on your laptop so you get `backend\models\gate\model.onnx`, then restart uvicorn.

In [ ]:
!cd out && zip -q -r ../gate_model.zip gate/model.onnx gate/labels.txt gate/metrics.json
files.download("gate_model.zip")